# Domain 1 – Tyre Degradation: Cluster Analysis

This notebook performs:
1. **K-means clustering** for driver style classification (Preserver / Balanced / Aggressor)
2. **Hierarchical clustering** for track regime classification (Low / Medium / High degradation)
3. PCA 2D projection for cluster visualisation
4. Driver style analysis


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.utils.paths import DOMAIN1_SILVER
print('Imports OK')

## 1. Load Data

In [ ]:
stints = pd.read_parquet(DOMAIN1_SILVER / 'stints_degradation.parquet')
print(f'Stints shape: {stints.shape}')

try:
    style_signals = pd.read_parquet(DOMAIN1_SILVER / 'driving_style_signals.parquet')
    print(f'Style signals shape: {style_signals.shape}')
except FileNotFoundError:
    style_signals = None
    print('driving_style_signals.parquet not found – run driving_style_signals.py first.')

## 2. Driver Style K-Means Clustering (k=3)

In [ ]:
if style_signals is not None and 'driving_style' in style_signals.columns:
    style_counts = style_signals['driving_style'].value_counts()
    print('Driver style distribution:')
    print(style_counts)

    signal_cols = [c for c in ['consistency_cv', 'deg_mgmt_score', 'exploit_score']
                   if c in style_signals.columns]

    if len(signal_cols) >= 2:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(style_signals[signal_cols].fillna(0))

        # Elbow plot
        inertias = []
        K_range = range(2, 8)
        for k in K_range:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            km.fit(X_scaled)
            inertias.append(km.inertia_)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2)
        axes[0].axvline(3, color='red', linestyle='--', label='k=3 selected')
        axes[0].set_title('Elbow Method: K-Means Inertia')
        axes[0].set_xlabel('Number of Clusters (k)')
        axes[0].set_ylabel('Inertia')
        axes[0].legend()

        # PCA scatter coloured by style
        pca = PCA(n_components=2, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        style_signals['pca1'] = X_pca[:, 0]
        style_signals['pca2'] = X_pca[:, 1]

        style_palette = {'Preserver': '#2ecc71', 'Balanced': '#3498db', 'Aggressor': '#e74c3c'}
        for style, group in style_signals.groupby('driving_style'):
            axes[1].scatter(group['pca1'], group['pca2'],
                           label=style, color=style_palette.get(style, 'grey'),
                           alpha=0.7, s=60)
            # Annotate driver labels
            for _, row in group.iterrows():
                axes[1].annotate(row.get('driver', ''), (row['pca1'], row['pca2']),
                                fontsize=7, alpha=0.8)

        axes[1].set_title(f'Driver Styles – PCA Projection\n(PC1: {pca.explained_variance_ratio_[0]:.1%}, PC2: {pca.explained_variance_ratio_[1]:.1%})')
        axes[1].set_xlabel('Principal Component 1')
        axes[1].set_ylabel('Principal Component 2')
        axes[1].legend(title='Driving Style')

        plt.tight_layout()
        plt.show()
    else:
        print('Insufficient signal columns for PCA.')
else:
    print('Style signals not available. Run driving_style_signals.py first.')

## 3. Hierarchical Clustering: Track Degradation Regimes

In [ ]:
event_col = 'event' if 'event' in stints.columns else 'EventName'
compound_col = 'compound' if 'compound' in stints.columns else 'tyre_compound'

# Build track-level degradation feature matrix
track_features = (
    stints.groupby(event_col).agg(
        mean_deg_linear=('deg_rate_linear', 'mean'),
        std_deg_linear=('deg_rate_linear', 'std'),
        mean_stint_length=('stint_length', 'mean'),
        mean_r2_linear=('r2_linear', 'mean'),
    ).reset_index().dropna()
)

if len(track_features) >= 3:
    feat_cols = ['mean_deg_linear', 'std_deg_linear', 'mean_stint_length', 'mean_r2_linear']
    X_track = track_features[feat_cols].fillna(0).to_numpy()
    scaler_track = StandardScaler()
    X_track_scaled = scaler_track.fit_transform(X_track)

    # Compute linkage for dendrogram
    Z = linkage(X_track_scaled, method='ward')

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Dendrogram
    dendrogram(
        Z,
        labels=track_features[event_col].tolist(),
        orientation='left',
        ax=axes[0],
        color_threshold=0.7 * max(Z[:, 2]),
        leaf_font_size=9,
    )
    axes[0].set_title('Hierarchical Clustering of Track Degradation Regimes\n(Ward linkage)')
    axes[0].set_xlabel('Distance')
    axes[0].axvline(0.7 * max(Z[:, 2]), color='red', linestyle='--',
                    label='Threshold (3 clusters)')
    axes[0].legend()

    # AgglomerativeClustering with k=3
    agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
    track_features['regime_cluster'] = agg.fit_predict(X_track_scaled)

    # Assign regime labels based on mean degradation rate
    regime_means = track_features.groupby('regime_cluster')['mean_deg_linear'].mean()
    regime_order = regime_means.sort_values().index.tolist()
    regime_map = {regime_order[0]: 'Low-Deg', regime_order[1]: 'Medium-Deg', regime_order[2]: 'High-Deg'}
    track_features['regime'] = track_features['regime_cluster'].map(regime_map)

    # PCA scatter of tracks
    pca_track = PCA(n_components=2, random_state=42)
    X_track_pca = pca_track.fit_transform(X_track_scaled)
    track_features['pca1'] = X_track_pca[:, 0]
    track_features['pca2'] = X_track_pca[:, 1]

    regime_palette = {'Low-Deg': '#2ecc71', 'Medium-Deg': '#f39c12', 'High-Deg': '#e74c3c'}
    for regime, group in track_features.groupby('regime'):
        axes[1].scatter(group['pca1'], group['pca2'],
                       label=regime, color=regime_palette.get(regime, 'grey'),
                       s=80, alpha=0.85)
        for _, row in group.iterrows():
            axes[1].annotate(
                row[event_col].replace(' Grand Prix', ''),
                (row['pca1'], row['pca2']),
                fontsize=7, alpha=0.9
            )

    axes[1].set_title('Track Regime PCA Projection')
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')
    axes[1].legend(title='Track Regime')

    plt.tight_layout()
    plt.show()

    print('Track regime classification:')
    print(track_features[[event_col, 'regime', 'mean_deg_linear']].sort_values('mean_deg_linear', ascending=False))
else:
    print(f'Need >= 3 tracks for hierarchical clustering, found {len(track_features)}.')

## 4. Driver Style vs Track Regime Interaction

In [ ]:
if style_signals is not None and 'driving_style' in style_signals.columns:
    style_counts = style_signals['driving_style'].value_counts()

    fig, ax = plt.subplots(figsize=(10, 6))
    style_palette = {'Preserver': '#2ecc71', 'Balanced': '#3498db', 'Aggressor': '#e74c3c'}
    style_counts.plot(
        kind='bar', ax=ax,
        color=[style_palette.get(s, 'grey') for s in style_counts.index]
    )
    ax.set_title('Driver Style Classification Distribution')
    ax.set_xlabel('Driving Style')
    ax.set_ylabel('Number of Drivers')
    ax.tick_params(axis='x', rotation=0)

    # Add driver names to each bar
    for style, group in style_signals.groupby('driving_style'):
        print(f'{style}: {", ".join(sorted(group["driver"].tolist()))}')

    plt.tight_layout()
    plt.show()